# Imports

GPT2LMHeadModel - GPT2 with additional "head" layer on top of language modelling. Traditional GPT2Model returns hidden state and GPT2LMHeadModel additionally returns linear layer at the end, which maps hidden states onto probability distribution over whole dictionary (logit for every possible next token).

Without that head we have only internal representation of the model (its useful for mech interpret or embeddings). With Head we have ready model to generate text - directly computes loss (cross-entropy loss) and we can call `.generate()`

In [1]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

In [2]:
# Load pre-trained GPT2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2-medium')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

It's common to set pad token to be end of sequence token

In [3]:
# GPT-2 does not set a pad token
print('Current pad token: ', tokenizer.pad_token)
print('Current eos token: ', tokenizer.eos_token)

# common (though not necessarily ideal) to set pad_token = eos_token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print('Current pad token: ', tokenizer.pad_token)

Current pad token:  None
Current eos token:  <|endoftext|>
Current pad token:  <|endoftext|>


# Encoding

In [5]:
sentences = [
    'Every happy family is the same',
    'every unhappy family is unhappy it its own way',
    'Why did the chicken cross the road'
]

# tokenize with padding and attention mask
toks = tokenizer(
    sentences,
    return_tensors = 'pt',
    padding = True
)
toks

{'input_ids': tensor([[ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256],
        [16833, 19283,  1641,   318, 19283,   340,   663,   898,   835],
        [ 5195,   750,   262,  9015,  3272,   262,  2975, 50256, 50256]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0]])}

In [7]:
toks['input_ids'][0][-1], tokenizer.decode(toks['input_ids'][0][-1])

(tensor(50256), '<|endoftext|>')

In [8]:
toks['attention_mask']

tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0, 0]])

In attention mask we can see that tokenizer took longer sentence and padded all other sentences with eos token OR 0 in attention mask up to the end of sequence.

In [9]:
for i in range(len(sentences)):

  inputs = toks['input_ids'][i]
  mask = toks['attention_mask'][i]

  print(f'"{sentences[i]}":')
  print(inputs, '\n', mask, '\n')

"Every happy family is the same":
tensor([ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256]) 
 tensor([1, 1, 1, 1, 1, 1, 0, 0, 0]) 

"every unhappy family is unhappy it its own way":
tensor([16833, 19283,  1641,   318, 19283,   340,   663,   898,   835]) 
 tensor([1, 1, 1, 1, 1, 1, 1, 1, 1]) 

"Why did the chicken cross the road":
tensor([ 5195,   750,   262,  9015,  3272,   262,  2975, 50256, 50256]) 
 tensor([1, 1, 1, 1, 1, 1, 1, 0, 0]) 



# Generate some text

In [10]:
outputs = model.generate(
    input_ids      = toks['input_ids'],        # inputs
    attention_mask = toks['attention_mask'],   # we dont need it necessarily unless we have multipple sequences with different sequence length
    pad_token_id   = tokenizer.pad_token_id,   # we can explicitly define padding token
    max_length     = 66,                       # max length of the output WHICH INCLUDES INPUT
    num_return_sequences = 1,                  #
    do_sample      = True,                     # probabilistically select next token not only the highest prob
    top_k          = 50,                       # select next token out of 50 top p tokens
    top_p          = .95,                      # select next token out of tokens with at least .95 p
)

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


In [20]:
outputs

tensor([[ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256,   262,
           835,   326,   262,   922,  1641,   318,   262,   976,   835,   326,
           340,   815,   307,    13,  1406,    11,   351,   345,    11,   319,
           262,   584,  1021,    11,   356,   765,   284,   307,  1498,   284,
           307,   517,  4497,   329,   674,  3988,    11,   329,   703,   356,
          5412,   511,  4069,    11,   703,   356,  3031,   284,   511,  4069,
            11,   703,   356,  2209,   606,   290],
        [16833, 19283,  1641,   318, 19283,   340,   663,   898,   835,   526,
           198,   198,  2202,   530,  1735,   318,   262,  5582,  2055,   286,
           520,    13,  2940,   338, 35637,    11,   543,   635,  7777,   257,
          5582,  2055, 27316,   287,   262,  1748,   338, 20390, 30379,  6232,
           290,  6716,    11,   262,  8108,  3337,    13,  1550,   262,   584,
           318,   262,  5582,  8108,  3337,    11,   530,   286,   734,  5582,


`batch_decode` used when we have multiple lines of sequences to decode. `decode` is used when we have 1 sequence to decode.

In [22]:
# decode
decoded_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

for i, text in enumerate(decoded_texts):
  print(f'*** Text {i+1}:\n{text}\n')

*** Text 1:
Every happy family is the same the way that the good family is the same way that it should be. So, with you, on the other hand, we want to be able to be more responsible for our kids, for how we handle their behavior, how we respond to their behavior, how we address them and

*** Text 2:
every unhappy family is unhappy it its own way."

On one side is the Jewish community of St. Mark's Cemetery, which also houses a Jewish community cemetery in the city's Upper Arlington neighborhood and nearby, the Community Center. On the other is the Jewish Community Center, one of two Jewish centers that was built to be

*** Text 3:
Why did the chicken cross the road-and why did it hit the fence? The chicken doesn't cross, it's not coming after you, not in your face. It's not coming up to you either, it's not going after you. This is a very simple part of the story of the show. Because



or simpler

In [23]:
model.generate(toks['input_ids'])

[transformers] The attention mask and the pad token id were not set, with a batched input. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
/usr/local/lib/python3.13/dist-packages/transformers/generation/utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=29) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


tensor([[ 6109,  3772,  1641,   318,   262,   976, 50256, 50256, 50256,   464,
           471,    13,    50,    13,  2732,   286,  4796,   468,  5717,   257,
          8087,  1028,   262,   471,    13,    50,    13,  2732,   286],
        [16833, 19283,  1641,   318, 19283,   340,   663,   898,   835,    13,
           198,   198,     1,    40,   892,   340,   338,   257,   845,  1593,
          2071,   329,   262,  1499,    13,   632,   338,   257,   845],
        [ 5195,   750,   262,  9015,  3272,   262,  2975, 50256, 50256,   464,
           471,  1546,    47, 32603,   784,  3406,  2723,   329,   383, 15624,
         28859,  1201,  8735,   198,   198,  1212,  2708,   318,   546]])